# Retrieval augmented Fine-tuning

In this notebook, we perform **fine-tuning on the pre-trained models** BERT and RoBERTa with our **retrieval-augmented inputs**.  

The retriever is always `sentence-transformers/all-mpnet-base-v2` (contrastive, shared across all configs).

For each selected combination, we follow this **training pipeline**:
1. Load the sbert retriever once (shared)
2. Retrieve k nearest neighbors from the chosen FAISS index (`training`, `documents`, or `full`), with self-exclusion at train time
3. Build an augmented input: `query [SEP] [hate] neighbor1 [SEP] [not hate] neighbor2 ...`
4. Fine-tune a classifier on the augmented inputs

**Configurable dimensions**:
| Variable | Options |
|---|---|
| `SELECTED_MODELS` | `bert`, `roberta` |
| `SELECTED_INDEX_TYPES` | `training`, `documents`, `full` |
| `SELECTED_DATASETS` | `IHC`, `ISHate` |

**Outputs:**
```
weights_rag/{model}/sbert/{index_type}/{dataset}/
```

## 1. Imports

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from rag import encode, retrieve_top_k_above_threshold

## 2. Configuration

Here you can choose which model to train. Modify : SELECTED_MODELS, ..., SELECTED_DATASETS as you like

In [2]:
RAG_DIR         = Path('.')
WEIGHTS_RAG_DIR = Path('..') / 'weights_rag'
INDEX_DIR       = RAG_DIR / 'index'

MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

# Contrastive retriever — shared across all classifier configs (not per-model-family)
RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# === What to run — edit these lists to select any subset ===
SELECTED_MODELS      = ['bert', 'roberta']  
SELECTED_INDEX_TYPES = ['training', 'documents', 'full']
SELECTED_DATASETS    = ['Vicomtech']

# Retrieval config
K         = 5
THRESHOLD = 0.4   # lower threshold: sbert embeddings are well-separated (not collapsed to ~0.997)

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device            : {device}')
print(f'Retriever         : {RETRIEVER_HF_ID}')
print(f'k / threshold     : {K} / {THRESHOLD}')
print(f'Models            : {SELECTED_MODELS}')
print(f'Index types       : {SELECTED_INDEX_TYPES}')
print(f'Datasets          : {SELECTED_DATASETS}')

Device            : cuda
Retriever         : sentence-transformers/all-mpnet-base-v2
k / threshold     : 5 / 0.4
Models            : ['bert', 'roberta']
Index types       : ['training', 'documents', 'full']
Datasets          : ['Vicomtech']


## 3. Load Datasets

Here we load IHC and/or ISHate and/or Vicomtech depending on the choosen datasets in `SELECTED_DATASETS`.  
Same train/test splits as in `baseline.ipynb`.

In [3]:
# IHC
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example['label'] = 0 if example['class'] == 'not_hate' else 1
    return example

train_ihc = splits['train'].map(add_binary_label_ihc)
test_ihc  = splits['test'].map(add_binary_label_ihc)

# ISHate
ishate_raw = load_dataset('BenjaminOcampo/ISHate')

def add_binary_label_ishate(example):
    example['label'] = 0 if example['hateful_layer'] == 'Non-HS' else 1
    return example

train_ishate = ishate_raw['train'].map(add_binary_label_ishate)
test_ishate  = ishate_raw['test'].map(add_binary_label_ishate)

# Vicomtech
import urllib.request, zipfile, shutil

_repo_dir      = str(RAG_DIR / 'data' / 'hate-speech-dataset')
_metadata_path = f'{_repo_dir}/annotations_metadata.csv'
_train_dir     = f'{_repo_dir}/sampled_train'
_test_dir      = f'{_repo_dir}/sampled_test'

if not all(os.path.exists(p) for p in [_metadata_path, _train_dir, _test_dir]):
    if os.path.isdir(_repo_dir):
        shutil.rmtree(_repo_dir)
    os.makedirs(str(RAG_DIR / 'data'), exist_ok=True)
    _zip_url  = 'https://github.com/Vicomtech/hate-speech-dataset/archive/refs/heads/master.zip'
    _zip_path = str(RAG_DIR / 'data' / 'hate-speech-dataset.zip')
    urllib.request.urlretrieve(_zip_url, _zip_path)
    with zipfile.ZipFile(_zip_path, 'r') as zf:
        zf.extractall(str(RAG_DIR / 'data'))
    os.rename(str(RAG_DIR / 'data' / 'hate-speech-dataset-master'), _repo_dir)
    os.remove(_zip_path)

_metadata = pd.read_csv(_metadata_path).set_index('file_id')

def _load_vicomtech_split(split_dir):
    rows = []
    for fname in sorted(os.listdir(split_dir)):
        if not fname.endswith('.txt'):
            continue
        file_id = fname[:-4]
        if file_id not in _metadata.index:
            continue
        label_str = _metadata.loc[file_id, 'label']
        if label_str not in ('hate', 'noHate'):
            continue
        with open(os.path.join(split_dir, fname), encoding='utf-8') as f:
            text = f.read().strip()
        rows.append({'text': text, 'label': 1 if label_str == 'hate' else 0})
    return Dataset.from_list(rows)

vicomtech_train = _load_vicomtech_split(_train_dir)
vicomtech_test  = _load_vicomtech_split(_test_dir)

# Dataset registry
DATASETS = {
    'IHC':       {'train': train_ihc,       'test': test_ihc,       'text_col': 'post'},
    'ISHate':    {'train': train_ishate,     'test': test_ishate,    'text_col': 'text'},
    'Vicomtech': {'train': vicomtech_train,  'test': vicomtech_test, 'text_col': 'text'},
}

print(f'IHC        — train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate     — train: {len(train_ishate):,}  test: {len(test_ishate):,}')
print(f'Vicomtech  — train: {len(vicomtech_train):,}  test: {len(vicomtech_test):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/55023 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

IHC        — train: 19,332  test: 2,148
ISHate     — train: 55,023  test: 4,368
Vicomtech  — train: 1,914  test: 478


## 4. Self-Exclusion Lookup

`chunks_training.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.

In [4]:
chunks_df = pd.read_csv(RAG_DIR / 'chunks' / 'chunks_training.csv')

def strip_label_prefix(text):
    return text.replace('[hate] ', '', 1).replace('[not hate] ', '', 1)

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 67,864 entries


## 5. Augmentation Function

Retrieves k neighbors for a dataset split using an explicit retriever (model/tokenizer/index).
Called once per model inside the training loop.

**Input format:** `{query} {sep} {[hate§/not hate] neighbor1} {sep} {[hate/not hate] neighbor2} ...`

- Query: **no** label prefix (that is what the model must predict).
- Neighbors: **keep** their `[hate]`/`[not hate]` prefix as few-shot context clues. No data leakage cause the index was made on training split and we apply a mask for self-exclusion.

In [5]:
def augment_split(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': [text for text, _ in neighbors],
            'label':     example['label'],
        })
    return records

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [6]:
def tokenize_augmented(records, tokenizer, max_length=MAX_LENGTH):
    sep = tokenizer.sep_token
    texts = [
        f' {sep} '.join([r['query']] + r['neighbors'])
        for r in records
    ]
    labels = [r['label'] for r in records]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    encoded['labels'] = labels
    return Dataset.from_dict(encoded)

## 7. Metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

## 8. Training Loop

For each selected combination of `(model, retriever_weight, index_type, dataset)`:
1. Load the retriever — either the base HuggingFace checkpoint or a fine-tuned checkpoint from `weights/`
2. Load the FAISS index (`vdb_{index_type}.faiss`) and its document lookup
3. Augment all selected datasets with that retriever + index
4. Free the retriever, then train a classifier for each dataset
5. Save weights to `weights_rag/{model}/{retriever_weight}/{index_type}/{dataset}/` and print classification report

**Recall: choose the models to train in section 2**

In [8]:
results = {}

# Load the single shared sbert retriever once
print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f"Retriever ready on {device}\n")

for index_type in SELECTED_INDEX_TYPES:
    # ── Load sbert FAISS index for this split ────────────────────────────────
    index_path = INDEX_DIR / 'sbert' / f'vdb_{index_type}.faiss'
    ret_index  = faiss.read_index(str(index_path))
    print(f"\n{'#'*60}")
    print(f"# Index: {index_type}  |  Vectors: {ret_index.ntotal:,}")
    print(f"{'#'*60}")

    with open(INDEX_DIR / f'lookup_{index_type}.json') as f:
        ret_documents = json.load(f)

    # ── Augment all selected datasets once per index split ───────────────────
    aug_data = {}
    for ds_name in SELECTED_DATASETS:
        ds_cfg = DATASETS[ds_name]
        print(f'\n=== Augmenting {ds_name} ===')
        aug_data[ds_name] = {
            'train': augment_split(ds_cfg['train'], ds_cfg['text_col'], True,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
            'test':  augment_split(ds_cfg['test'],  ds_cfg['text_col'], False,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
        }

    # ── Train a classifier for each (model, dataset) combo ──────────────────
    for model_name, hf_id in MODELS.items():
        if model_name not in SELECTED_MODELS:
            continue

        for ds_name in SELECTED_DATASETS:
            key = (model_name, index_type, ds_name)
            print(f"\n{'='*60}")
            print(f"Model: {model_name}  |  Index: {index_type}  |  Dataset: {ds_name}")
            print(f"{'='*60}")

            tokenizer = AutoTokenizer.from_pretrained(hf_id)
            tok_train = tokenize_augmented(aug_data[ds_name]['train'], tokenizer)
            tok_test  = tokenize_augmented(aug_data[ds_name]['test'],  tokenizer)

            model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

            # Output path uses 'sbert' to identify the retriever used
            save_path = str(WEIGHTS_RAG_DIR / model_name / 'sbert' / index_type / ds_name)
            os.makedirs(save_path, exist_ok=True)

            training_args = TrainingArguments(
                output_dir=f'./checkpoints_rag/{model_name}/sbert/{index_type}/{ds_name}',
                num_train_epochs=NUM_EPOCHS,
                per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=BATCH_SIZE * 2,
                learning_rate=LEARNING_RATE,
                eval_strategy='epoch',
                save_strategy='no',
                logging_strategy='epoch',
                report_to='none',
                seed=42,
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tok_train,
                eval_dataset=tok_test,
                compute_metrics=compute_metrics,
            )

            trainer.train()
            trainer.save_model(save_path)
            tokenizer.save_pretrained(save_path)
            print(f'  Weights saved → {save_path}')

            preds_out = trainer.predict(tok_test)
            preds  = np.argmax(preds_out.predictions, axis=-1)
            labels = [r['label'] for r in aug_data[ds_name]['test']]
            print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

            results[key] = {
                'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
                'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
                'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
            }

            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()

Loading retriever: sentence-transformers/all-mpnet-base-v2 ...


tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

              precision    recall  f1-score   support

      Non-HS       0.89      0.83      0.86       239
          HS       0.84      0.90      0.87       239

    accuracy                           0.86       478
   macro avg       0.86      0.86      0.86       478
weighted avg       0.86      0.86      0.86       478


Model: roberta  |  Index: full  |  Dataset: Vicomtech


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.520800,0.384308,0.826397,0.844773,0.828452
2,0.321000,0.328750,0.846257,0.856781,0.847280
3,0.200600,0.412231,0.826986,0.839975,0.828452


  Weights saved → ../weights_rag/roberta/sbert/full/Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.90      0.74      0.81       239
          HS       0.78      0.92      0.84       239

    accuracy                           0.83       478
   macro avg       0.84      0.83      0.83       478
weighted avg       0.84      0.83      0.83       478



## 9. Results

One table per training dataset. Rows = `model / retriever_weight / index_type`.

In [9]:
metric_labels = {'macro_f1': 'F1', 'macro_p': 'Precision', 'macro_r': 'Recall'}

rows = {}
for (m, it, ds), vals in results.items():
    row_key = f"{m}/sbert/{it}"
    if row_key not in rows:
        rows[row_key] = {}
    for metric, label in metric_labels.items():
        rows[row_key][(ds, label)] = vals[metric]

df = pd.DataFrame(rows).T
df.columns = pd.MultiIndex.from_tuples(df.columns)
df.index.name = 'Model / Retriever / Index'

styled = (
    df.style
    .format('{:.3f}')
    .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
    .set_caption('RAG Fine-tuning results — sbert retriever')
)
display(styled)